# 04 - Architectures, parameters, and losses

**Goals:** build models with Sequential and nn.Module, inspect parameters, and match losses to tasks.

Batches from [03](03_data_and_batches.ipynb) have shape `(B, 3)`; this notebook maps them to two class scores per row using a small artificial batch. Run cells in order in a fresh kernel; each notebook supplies its own setup.

## Build a classifier with `Sequential`

Set a seed to reproduce the initial parameters and example inputs when running the notebook from the beginning.

In [ ]:
import torch
from torch import nn

torch.manual_seed(486)


The first linear layer maps three features to eight hidden values; the final layer maps those to two logits. Between them, ReLU replaces negative values with zero, allowing the network to represent nonlinear relationships.

In [ ]:
model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
print(model)


Create a batch of four observations and one integer class label per observation. Each label identifies one of the two output classes.

In [ ]:
inputs = torch.randn(4, 3)  # Four observations, three features each.
labels = torch.tensor([0, 1, 0, 1], dtype=torch.long)
print("input shape:", inputs.shape)
print("label shape:", labels.shape)


Follow the shape change through the hidden layer. Applying a linear layer to a batch applies the same weights and bias to every row.

In [ ]:
hidden = model[0](inputs)
activated = model[1](hidden)
print("hidden and activated shapes:", hidden.shape, activated.shape)


`model(inputs)` runs all three layers in order. It returns two unrestricted logits per observation.

In [ ]:
logits = model(inputs)
print("logits:\n", logits)
print("logits shape:", logits.shape)


## Inspect the parameters

Each linear layer has a weight matrix and a bias vector. The weight shape is `(output features, input features)`; ReLU has no learnable parameters.

Unlike the hand-built `Linear` in notebook 02, `nn.Linear` stores weights as `(outputs, inputs)` and computes `x @ weight.T + bias`.

In [ ]:
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))


Count all trainable scalar entries. For example, the first layer has 8 * 3 weights and 8 biases. The second has `2 * 8 + 2`, for 50 parameters altogether.

In [ ]:
print("number of trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


## Classification loss and predictions

`CrossEntropyLoss` combines log-softmax and negative log-likelihood. Give it raw logits of shape `(batch, classes)` and integer labels of shape `(batch,)`. By default it averages the loss over the batch.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
print("loss:", loss.item())


Use softmax across the class dimension when probabilities are needed: each row sums to one. The largest logit identifies the predicted class; applying softmax first would give the same class.

In [ ]:
probabilities = torch.softmax(logits, dim=1)
print("probabilities:\n", probabilities)
print("row sums:", probabilities.sum(dim=1))
print("predicted classes:", logits.argmax(dim=1))


## Define a custom module

A custom module lets us name layers and describe more complex computations. Define layers in `__init__` and their computation in `forward`. Calling `custom_model(inputs)` runs `forward` through PyTorch's module machinery.

In [ ]:
class Classifier(nn.Module):
    def __init__(self, n_features, hidden_units, n_classes):
        super().__init__()
        self.hidden = nn.Linear(n_features, hidden_units)
        self.output = nn.Linear(hidden_units, n_classes)

    def forward(self, x):
        x = torch.relu(self.hidden(x))
        return self.output(x)


In [ ]:
custom_model = Classifier(3, 8, 2)
print("custom output shape:", custom_model(inputs).shape)


The custom model has the same layer sizes as the `Sequential` model, but its parameters are initialized separately, so its logits generally differ.

## Regression and mean squared error

For regression, this linear model returns one continuous prediction per observation. Use floating-point targets with the same `(batch, 1)` shape so each prediction is compared with its corresponding target.

In [ ]:
regression_model = nn.Linear(3, 1)
targets = torch.randn(4, 1)
predictions = regression_model(inputs)
print("prediction and target shapes:", predictions.shape, targets.shape)


Mean squared error averages the squared differences between predictions and targets. This value measures the error before any training. This is the built-in version of the squared-error objective in notebook 02.

In [ ]:
mse = nn.MSELoss()(predictions, targets)
print("regression MSE:", mse.item())


These cells compute losses and predictions with initialized parameters. Training will add gradient calculation and parameter updates in the next notebook.

## Try it

1. Change the hidden width and calculate the parameter count by hand.

2. Add a second hidden layer to `Classifier` and confirm the output shape.

3. Compare raw logits, predicted classes, and probabilities. Why is softmax not passed into `CrossEntropyLoss`?

4. What goes wrong if regression predictions have shape `(4, 1)` but targets have shape `(4,)`?

**Next:** [Training and evaluation](05_training_and_evaluation.ipynb) combines the loader, model, loss, and gradients into a training loop.